In [0]:
# window functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Read the Dataframe

df = spark.read.format("csv").option("header", "true").option("inferSchema", True).load("/FileStore/tables/tallest_people_in_the_world.csv")
display(df)

In [0]:
# I want to find out the 2nd tallest person in each country

window_spec = Window.partitionBy("country").orderBy(F.col("height_cm").desc())
new_df = (
    df
    .withColumn("rn", F.dense_rank().over(window_spec))
    .filter(F.col("rn") == 2)
    .drop("rn")
)

new_df.display()

In [0]:
# create a temp table
df.createOrReplaceTempView("tallest_people")

In [0]:
%sql
select * from tallest_people;

In [0]:
%sql
select 
*, 
dense_rank() over (partition by country order by height_cm desc) as rk
from tallest_people
where rk = 2;

In [0]:
%sql
select 
*
from tallest_people
qualify dense_rank() over (partition by country order by height_cm desc) = 2;

In [0]:
%sql
select * from tallest_people limit 5;

In [0]:
# Read a table in samples database

cus_df = spark.table("samples.tpch.customer")
cus_df.display()

In [0]:
orders_df = spark.table("samples.tpch.orders")
orders_df.display()

In [0]:
# Joins

In [0]:
# Find the customer who have placed most no.of orders

added_cus_df = (
    cus_df
    .select(
        F.col("c_custkey").alias("CustomerID"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        orders_df
        .select(
            F.col("o_custkey").alias("CustomerID"),
            F.col("o_orderkey").alias("OrderID")
        ),
        on=["CustomerID"],
        how="inner"
    )

)

added_cus_df.display()

In [0]:
val_cus_df = (
    added_cus_df
    .groupBy("CustomerID", "CustomerName")
    .agg(F.count(F.col("OrderID")).alias("OrdersCount"))
    .orderBy(F.col("OrdersCount").desc())
    .limit(1)
)
val_cus_df.display()

In [0]:
# Find out the customers who have not ordered anything yet

# Find the customer who have placed most no.of orders

non_cus_df = (
    cus_df
    .select(
        F.col("c_custkey"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        orders_df
        .select(
            F.col("o_custkey"),
            F.col("o_orderkey").alias("OrderID")
        ),
        on=cus_df.c_custkey == orders_df.o_custkey,
        how="leftanti"
    )

)

non_cus_df.display()

In [0]:
# Find out the customers who have not ordered anything yet

# Find the customer who have placed most no.of orders

non_cus_df = (
    cus_df
    .select(
        F.col("c_custkey"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        orders_df
        .select(
            F.col("o_custkey"),
            F.col("o_orderkey").alias("OrderID")
        ),
        on=cus_df.c_custkey == orders_df.o_custkey,
        how="left"
    )

)

non_cus_df.display()